In [1]:
# Import the tools we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

print("="*50)
print("STUDENT PERFORMANCE PREDICTION")
print("="*50)

# Load the data we created in Day 1
try:
    df = pd.read_csv('student_performance.csv')
    print("\n✅ Loaded student data from Day 1")
except:
    # If Day 1 data doesn't exist, create simple data
    print("\n📝 Creating sample student data...")
    np.random.seed(42)
    
    # Create 100 students with random study hours
    df = pd.DataFrame({
        'hours_studied': np.random.randint(1, 12, 100),
        'previous_score': np.random.randint(50, 95, 100),
        'attendance': np.random.randint(60, 100, 100),
        'final_score': 0  # Will calculate below
    })
    
    # Calculate final score (more study hours = higher score)
    df['final_score'] = (df['hours_studied'] * 3.5 + df['previous_score'] * 0.4 + df['attendance'] * 0.2 + np.random.normal(0, 5, 100))
    df['final_score'] = df['final_score'].clip(40, 100).round(1)

print(f"\n📊 We have data for {len(df)} students")
print(f"📋 Columns: {list(df.columns)}")
print("\nFirst 5 students:")
print(df.head())


STUDENT PERFORMANCE PREDICTION

✅ Loaded student data from Day 1

📊 We have data for 500 students
📋 Columns: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular', 'parent_education', 'internet_access', 'final_score', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']

First 5 students:
   student_id  hours_studied  previous_score  attendance  sleep_hours  \
0           1              7              97          72          4.2   
1           2              4              97          64          5.3   
2           3             13              61          86          5.4   
3           4             11              88          70          8.0   
4           5              8              91          63          4.1   

  extracurricular parent_education internet_access  final_score  \
0             Yes           Master              No        86.71   
1              No         Bachelor             Yes        87.98  

In [4]:
iris_df = pd.read_csv('./data/iris.csv')
print("\n Loaded Iris dataset")
print(f"\n we have data for {len(iris_df)} iris samples")

print("\nFirst 5 iris samples:")
print(iris_df.head())

print("\n Iris Columns:", list(iris_df.columns))

print("\n Statistics of Iris Dataset:")
print(iris_df.describe())



 Loaded Iris dataset

 we have data for 150 iris samples

First 5 iris samples:
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

 Iris Columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

 Statistics of Iris Dataset:
       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.054000      3.758667     1.198667
std        0.828066     0.433594      1.764420     0.763161
min        4.300000     2.000000      1.000000     0.100000
25%        5.100000     2.800000      1.600000     0.300000
50%        5.800000     3.000000      4.350000     1.30

In [5]:
def validate_data(df):
    """
    Real-world data validation function
    Checks for common data quality issues
    """
    issues = []
    
    print("🔍 DATA VALIDATION REPORT")
    print("="*40)
    
    # 1. Check missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        issues.append(f"Missing values found: {missing[missing>0].to_dict()}")
        print(f"⚠️ Missing values: {missing[missing>0]}")
    else:
        print("✅ No missing values")
    
    # 2. Check duplicates
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        issues.append(f"Duplicate rows: {duplicates}")
        print(f"⚠️ Duplicate rows: {duplicates}")
    else:
        print("✅ No duplicate rows")
    
    # 3. Check numeric ranges
    print("\n📊 Range Validation:")
    for col in ['hours_studied', 'previous_score', 'attendance', 'sleep_hours']:
        if col in df.columns:
            min_val = df[col].min()
            max_val = df[col].max()
            print(f"  {col}: {min_val} - {max_val}")
            
            # Check for impossible values
            if col == 'attendance' and max_val > 100:
                issues.append(f"Attendance > 100% found")
            if col == 'sleep_hours' and max_val > 24:
                issues.append(f"Impossible sleep hours: {max_val}")
    
    # 4. Check target variable
    if 'final_score' in df.columns:
        print(f"\n  final_score range: {df['final_score'].min()} - {df['final_score'].max()}")
        if df['final_score'].min() < 0 or df['final_score'].max() > 100:
            issues.append("Final score outside 0-100 range")
    
    # 5. Check categorical consistency
    print("\n🏷️ Categorical Value Consistency:")
    for col in ['extracurricular', 'internet_access']:
        if col in df.columns:
            unique = df[col].unique()
            print(f"  {col}: {unique}")
            if not set(unique).issubset({'Yes', 'No'}):
                issues.append(f"Unexpected values in {col}: {unique}")
    
    if issues:
        print("\n" + "="*40)
        print("⚠️ ISSUES DETECTED:")
        for issue in issues:
            print(f"  • {issue}")
    else:
        print("\n✅ All validation checks passed!")
    
    return len(issues) == 0

is_valid = validate_data(df)
print("\n" + "="*60)

🔍 DATA VALIDATION REPORT
✅ No missing values
✅ No duplicate rows

📊 Range Validation:
  hours_studied: 1 - 14
  previous_score: 40 - 99
  attendance: 50 - 99
  sleep_hours: 4.0 - 10.0

  final_score range: 40.79 - 100.0

🏷️ Categorical Value Consistency:
  extracurricular: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
  internet_access: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

✅ All validation checks passed!



In [6]:
def perform_eda(df):
    """
    Professional EDA function for real-world projects
    """
    print("="*60)
    print("EXPLORATORY DATA ANALYSIS")
    print("="*60)
    
    # 1. Statistical summary
    print("\n1. STATISTICAL SUMMARY")
    print("-"*40)
    print(df.describe())
    
    # 2. Distribution analysis
    print("\n2. DISTRIBUTION ANALYSIS")
    print("-"*40)
    for col in ['hours_studied', 'attendance', 'final_score']:
        skewness = df[col].skew()
        print(f"  {col}: skewness = {skewness:.3f}", end=" ")
        if abs(skewness) > 1:
            print("(Highly skewed - may need transformation)")
        elif abs(skewness) > 0.5:
            print("(Moderately skewed)")
        else:
            print("(Approximately symmetric)")
    
    # 3. Outlier detection
    print("\n3. OUTLIER DETECTION (IQR Method)")
    print("-"*40)
    for col in ['hours_studied', 'attendance', 'sleep_hours', 'final_score']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]
        print(f"  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")
    
    # 4. Correlation analysis
    print("\n4. CORRELATION WITH TARGET")
    print("-"*40)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    correlations = df[numeric_cols].corr()['final_score'].sort_values(ascending=False)
    for col, corr in correlations.items():
        if col != 'final_score':
            strength = "Strong" if abs(corr) > 0.7 else "Moderate" if abs(corr) > 0.3 else "Weak"
            direction = "positive" if corr > 0 else "negative"
            print(f"  {col}: {corr:.3f} ({direction}, {strength})")
    
    # 5. Group analysis
    print("\n5. CATEGORICAL IMPACT")
    print("-"*40)
    for col in ['extracurricular', 'internet_access', 'parent_education']:
        if col in df.columns:
            means = df.groupby(col)['final_score'].mean().sort_values(ascending=False)
            print(f"\n  {col}:")
            for cat, val in means.items():
                print(f"    {cat}: {val:.2f}")

perform_eda(df)

EXPLORATORY DATA ANALYSIS

1. STATISTICAL SUMMARY
----------------------------------------
       student_id  hours_studied  previous_score  attendance  sleep_hours  \
count  500.000000     500.000000      500.000000  500.000000   500.000000   
mean   250.500000       7.444000       70.356000   74.246000     7.030200   
std    144.481833       4.113966       17.500222   14.438227     1.784207   
min      1.000000       1.000000       40.000000   50.000000     4.000000   
25%    125.750000       4.000000       55.000000   61.000000     5.400000   
50%    250.500000       7.000000       71.000000   75.000000     7.100000   
75%    375.250000      11.000000       86.000000   86.250000     8.600000   
max    500.000000      14.000000       99.000000   99.000000    10.000000   

       final_score  extracurricular_encoded  parent_education_encoded  \
count    500.00000               500.000000                500.000000   
mean      88.27742                 0.394000                  1.530000

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("="*60)
print("DAY 2: REAL-WORLD LINEAR REGRESSION")
print("="*60)

# Load data from Day 1
try:
    df = pd.read_csv('student_performance.csv')
    print("\n✅ Loaded data from Day 1")
except:
    print("\n⚠️ Creating sample data (run Day 1 first for real data)")
    np.random.seed(42)
    n = 500
    df = pd.DataFrame({
        'student_id': range(1, n+1),
        'hours_studied': np.random.randint(1, 15, n),
        'previous_score': np.random.randint(40, 100, n),
        'attendance': np.random.randint(50, 100, n),
        'sleep_hours': np.random.uniform(4, 10, n).round(1),
        'extracurricular': np.random.choice(['Yes', 'No'], n, p=[0.4, 0.6]),
        'parent_education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
        'internet_access': np.random.choice(['Yes', 'No'], n, p=[0.8, 0.2])
    })
    df['final_score'] = (
        df['hours_studied'] * 2.5 + df['previous_score'] * 0.5 +
        df['attendance'] * 0.3 + df['sleep_hours'] * 1.5 +
        (df['extracurricular'] == 'Yes') * 5 + (df['internet_access'] == 'Yes') * 3 +
        np.random.normal(0, 5, n)
    ).clip(0, 100).round(2)

print(f"\n📊 Dataset Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

DAY 2: REAL-WORLD LINEAR REGRESSION

✅ Loaded data from Day 1

📊 Dataset Shape: (500, 12)
📋 Columns: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular', 'parent_education', 'internet_access', 'final_score', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']


DAY 2: REAL-WORLD LINEAR REGRESSION

✅ Loaded data from Day 1

📊 Dataset Shape: (500, 12)
📋 Columns: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular', 'parent_education', 'internet_access', 'final_score', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']


In [5]:
def validate_data(df):
    """
    Real-world data validation function
    Checks for common data quality issues
    """
    issues = []
    
    print("🔍 DATA VALIDATION REPORT")
    print("="*40)
    
    # 1. Check missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        issues.append(f"Missing values found: {missing[missing>0].to_dict()}")
        print(f"⚠️ Missing values: {missing[missing>0]}")
    else:
        print("✅ No missing values")
    
    # 2. Check duplicates
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        issues.append(f"Duplicate rows: {duplicates}")
        print(f"⚠️ Duplicate rows: {duplicates}")
    else:
        print("✅ No duplicate rows")
    
    # 3. Check numeric ranges
    print("\n📊 Range Validation:")
    for col in ['hours_studied', 'previous_score', 'attendance', 'sleep_hours']:
        if col in df.columns:
            min_val = df[col].min()
            max_val = df[col].max()
            print(f"  {col}: {min_val} - {max_val}")
            
            # Check for impossible values
            if col == 'attendance' and max_val > 100:
                issues.append(f"Attendance > 100% found")
            if col == 'sleep_hours' and max_val > 24:
                issues.append(f"Impossible sleep hours: {max_val}")
    
    # 4. Check target variable
    if 'final_score' in df.columns:
        print(f"\n  final_score range: {df['final_score'].min()} - {df['final_score'].max()}")
        if df['final_score'].min() < 0 or df['final_score'].max() > 100:
            issues.append("Final score outside 0-100 range")
    
    # 5. Check categorical consistency
    print("\n🏷️ Categorical Value Consistency:")
    for col in ['extracurricular', 'internet_access']:
        if col in df.columns:
            unique = df[col].unique()
            print(f"  {col}: {unique}")
            if not set(unique).issubset({'Yes', 'No'}):
                issues.append(f"Unexpected values in {col}: {unique}")
    
    if issues:
        print("\n" + "="*40)
        print("⚠️ ISSUES DETECTED:")
        for issue in issues:
            print(f"  • {issue}")
    else:
        print("\n✅ All validation checks passed!")
    
    return len(issues) == 0

is_valid = validate_data(df)
print("\n" + "="*60)

🔍 DATA VALIDATION REPORT
✅ No missing values
✅ No duplicate rows

📊 Range Validation:
  hours_studied: 1 - 14
  previous_score: 40 - 99
  attendance: 50 - 99
  sleep_hours: 4.0 - 10.0

  final_score range: 40.79 - 100.0

🏷️ Categorical Value Consistency:
  extracurricular: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
  internet_access: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

✅ All validation checks passed!



In [6]:
def perform_eda(df):
    """
    Professional EDA function for real-world projects
    """
    print("="*60)
    print("EXPLORATORY DATA ANALYSIS")
    print("="*60)
    
    # 1. Statistical summary
    print("\n1. STATISTICAL SUMMARY")
    print("-"*40)
    print(df.describe())
    
    # 2. Distribution analysis
    print("\n2. DISTRIBUTION ANALYSIS")
    print("-"*40)
    for col in ['hours_studied', 'attendance', 'final_score']:
        skewness = df[col].skew()
        print(f"  {col}: skewness = {skewness:.3f}", end=" ")
        if abs(skewness) > 1:
            print("(Highly skewed - may need transformation)")
        elif abs(skewness) > 0.5:
            print("(Moderately skewed)")
        else:
            print("(Approximately symmetric)")
    
    # 3. Outlier detection
    print("\n3. OUTLIER DETECTION (IQR Method)")
    print("-"*40)
    for col in ['hours_studied', 'attendance', 'sleep_hours', 'final_score']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]
        print(f"  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")
    
    # 4. Correlation analysis
    print("\n4. CORRELATION WITH TARGET")
    print("-"*40)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    correlations = df[numeric_cols].corr()['final_score'].sort_values(ascending=False)
    for col, corr in correlations.items():
        if col != 'final_score':
            strength = "Strong" if abs(corr) > 0.7 else "Moderate" if abs(corr) > 0.3 else "Weak"
            direction = "positive" if corr > 0 else "negative"
            print(f"  {col}: {corr:.3f} ({direction}, {strength})")
    
    # 5. Group analysis
    print("\n5. CATEGORICAL IMPACT")
    print("-"*40)
    for col in ['extracurricular', 'internet_access', 'parent_education']:
        if col in df.columns:
            means = df.groupby(col)['final_score'].mean().sort_values(ascending=False)
            print(f"\n  {col}:")
            for cat, val in means.items():
                print(f"    {cat}: {val:.2f}")

perform_eda(df)

EXPLORATORY DATA ANALYSIS

1. STATISTICAL SUMMARY
----------------------------------------
       student_id  hours_studied  previous_score  attendance  sleep_hours  \
count  500.000000     500.000000      500.000000  500.000000   500.000000   
mean   250.500000       7.444000       70.356000   74.246000     7.030200   
std    144.481833       4.113966       17.500222   14.438227     1.784207   
min      1.000000       1.000000       40.000000   50.000000     4.000000   
25%    125.750000       4.000000       55.000000   61.000000     5.400000   
50%    250.500000       7.000000       71.000000   75.000000     7.100000   
75%    375.250000      11.000000       86.000000   86.250000     8.600000   
max    500.000000      14.000000       99.000000   99.000000    10.000000   

       final_score  extracurricular_encoded  parent_education_encoded  \
count    500.00000               500.000000                500.000000   
mean      88.27742                 0.394000                  1.530000

In [11]:
class DataPreprocessor:
    """
    Production-ready data preprocessing class
    Saves all transformations for consistent application
    """
    
    def __init__(self):
        self.label_encoders = {}
        self.scaler = None
        self.feature_columns = None
        self.categorical_cols = None
        self.numeric_cols = None
    
    def fit(self, df, target_col='final_score'):
        """Fit preprocessor on training data"""
        
        # Identify column types
        self.categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        
        # Remove target from features
        if target_col in self.numeric_cols:
            self.numeric_cols.remove(target_col)
        
        print("Column Identification:")
        print(f"  Numeric features: {self.numeric_cols}")
        print(f"  Categorical features: {self.categorical_cols}")
        
        # Fit label encoders for categorical columns
        print("\nEncoding categorical variables:")
        for col in self.categorical_cols:
            self.label_encoders[col] = LabelEncoder()
            self.label_encoders[col].fit(df[col])
            mapping = dict(zip(self.label_encoders[col].classes_, 
                              self.label_encoders[col].transform(self.label_encoders[col].classes_)))
            print(f"  {col}: {mapping}")
        
        # Store feature columns
        encoded_cols = [col + '_encoded' for col in self.categorical_cols]
        self.feature_columns = self.numeric_cols + encoded_cols
        
        # Fit scaler
        self.scaler = StandardScaler()
        
        # Create sample transformed data to fit scaler
        sample_df = self.transform(df, fit_scaler=True)
        self.scaler.fit(sample_df[self.feature_columns])
        
        print(f"\n✅ Preprocessor fitted successfully")
        print(f"   Total features: {len(self.feature_columns)}")
        
        return self
    
    def transform(self, df, fit_scaler=False):
        """Transform data using fitted preprocessor"""
        df_transformed = df.copy()
        
        # Encode categorical variables
        for col in self.categorical_cols:
            if col in df_transformed.columns:
                # Handle unseen categories
                df_transformed[col + '_encoded'] = df_transformed[col].apply(
                    lambda x: self.label_encoders[col].transform([x])[0] 
                    if x in self.label_encoders[col].classes_ else -1
                )
        
        return df_transformed
    
    def fit_transform(self, df, target_col='final_score'):
        """Fit and transform in one step"""
        self.fit(df, target_col)
        return self.transform(df)
    
    def get_feature_columns(self):
        """Return feature columns for model training"""
        return self.feature_columns

# Initialize and fit preprocessor
preprocessor = DataPreprocessor()
df_processed = preprocessor.fit_transform(df)

print("\n" + "="*60)
print("Processed DataFrame Preview:")
print(df_processed[preprocessor.get_feature_columns() + ['final_score']].head())

Column Identification:
  Numeric features: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']
  Categorical features: ['extracurricular', 'parent_education', 'internet_access']

Encoding categorical variables:
  extracurricular: {'No': np.int64(0), 'Yes': np.int64(1)}
  parent_education: {'Bachelor': np.int64(0), 'High School': np.int64(1), 'Master': np.int64(2), 'PhD': np.int64(3)}
  internet_access: {'No': np.int64(0), 'Yes': np.int64(1)}


DuplicateError: Expected unique column names, got:
- 'extracurricular_encoded' 2 times
- 'parent_education_encoded' 2 times
- 'internet_access_encoded' 2 times

In [ ]:
class StudentPerformanceModel:
    """
    Production-ready ML model class
    Includes training, evaluation, and prediction capabilities
    """
    
    def __init__(self):
        self.model = None
        self.preprocessor = None
        self.scaler = None
        self.feature_columns = None
        self.is_trained = False
        self.training_metrics = {}
    
    def prepare_data(self, df, target_col='final_score'):
        """Prepare data for training"""
        
        # Initialize preprocessor if not exists
        if self.preprocessor is None:
            self.preprocessor = DataPreprocessor()
            df_processed = self.preprocessor.fit_transform(df, target_col)
        else:
            df_processed = self.preprocessor.transform(df)
        
        # Get feature columns
        self.feature_columns = self.preprocessor.get_feature_columns()
        
        # Prepare X and y
        X = df_processed[self.feature_columns]
        y = df_processed[target_col]
        
        return X, y
    
    def train(self, df, test_size=0.2, random_state=42):
        """Train the model with proper validation"""
        
        print("="*60)
        print("MODEL TRAINING")
        print("="*60)
        
        # Prepare data
        X, y = self.prepare_data(df)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        print(f"\n📊 Data Split:")
        print(f"   Training: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
        print(f"   Testing: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")
        
        # Scale features
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train model
        self.model = LinearRegression()
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_train_pred = self.model.predict(X_train_scaled)
        y_test_pred = self.model.predict(X_test_scaled)
        
        self.training_metrics = {
            'train_r2': r2_score(y_train, y_train_pred),
            'test_r2': r2_score(y_test, y_test_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred)
        }
        
        print(f"\n📈 Performance Metrics:")
        print(f"   Training R²: {self.training_metrics['train_r2']:.4f}")
        print(f"   Testing R²: {self.training_metrics['test_r2']:.4f}")
        print(f"   Testing RMSE: {self.training_metrics['test_rmse']:.2f}")
        print(f"   Testing MAE: {self.training_metrics['test_mae']:.2f}")
        
        # Check for overfitting
        r2_diff = self.training_metrics['train_r2'] - self.training_metrics['test_r2']
        if r2_diff > 0.1:
            print(f"\n⚠️ Possible overfitting detected! (R² diff: {r2_diff:.3f})")
        else:
            print(f"\n✅ No significant overfitting detected")
        
        # Cross-validation
        cv_scores = cross_val_score(self.model, X_train_scaled, y_train, cv=5, scoring='r2')
        print(f"\n📊 Cross-Validation (5-fold):")
        print(f"   Mean R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        self.is_trained = True
        
        # Store test data for later evaluation
        self.X_test = X_test
        self.y_test = y_test
        self.y_test_pred = y_test_pred
        
        return self.model
    
    def predict(self, student_data):
        """
        Make prediction for a single student
        
        Args:
            student_data: dict with feature values
        
        Returns:
            predicted_score: float
        """
        if not self.is_trained:
            raise Exception("Model not trained yet!")
        
        # Convert to DataFrame
        input_df = pd.DataFrame([student_data])
        
        # Preprocess
        input_processed = self.preprocessor.transform(input_df)
        
        # Ensure all features exist
        for col in self.feature_columns:
            if col not in input_processed.columns:
                input_processed[col] = 0
        
        # Select and scale features
        X_input = input_processed[self.feature_columns]
        X_input_scaled = self.scaler.transform(X_input)
        
        # Predict
        prediction = self.model.predict(X_input_scaled)[0]
        
        return prediction
    
    def predict_batch(self, students_data):
        """Make predictions for multiple students"""
        return [self.predict(student) for student in students_data]
    
    def get_feature_importance(self):
        """Get feature importance (coefficients)"""
        if not self.is_trained:
            raise Exception("Model not trained yet!")
        
        importance_df = pd.DataFrame({
            'Feature': self.feature_columns,
            'Coefficient': self.model.coef_
        }).sort_values('Coefficient', key=abs, ascending=False)
        
        return importance_df
    
    def save(self, filepath='student_model.pkl'):
        """Save model and all components"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'model': self.model,
                'preprocessor': self.preprocessor,
                'scaler': self.scaler,
                'feature_columns': self.feature_columns,
                'is_trained': self.is_trained,
                'training_metrics': self.training_metrics
            }, f)
        print(f"✅ Model saved to {filepath}")
    
    def load(self, filepath='student_model.pkl'):
        """Load saved model"""
        with open(filepath, 'rb') as f:
            saved = pickle.load(f)
        
        self.model = saved['model']
        self.preprocessor = saved['preprocessor']
        self.scaler = saved['scaler']
        self.feature_columns = saved['feature_columns']
        self.is_trained = saved['is_trained']
        self.training_metrics = saved['training_metrics']
        print(f"✅ Model loaded from {filepath}")
        return self

# Train the model
model = StudentPerformanceModel()
model.train(df)

Column Identification:
  Numeric features: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']
  Categorical features: ['extracurricular', 'parent_education', 'internet_access']

Encoding categorical variables:
  extracurricular: {'No': np.int64(0), 'Yes': np.int64(1)}
  parent_education: {'Bachelor': np.int64(0), 'High School': np.int64(1), 'Master': np.int64(2), 'PhD': np.int64(3)}
  internet_access: {'No': np.int64(0), 'Yes': np.int64(1)}


DuplicateError: Expected unique column names, got:
- 'extracurricular_encoded' 2 times
- 'parent_education_encoded' 2 times
- 'internet_access_encoded' 2 times

In [8]:
class StudentPerformanceModel:
    """
    Production-ready ML model class
    Includes training, evaluation, and prediction capabilities
    """
    
    def __init__(self):
        self.model = None
        self.preprocessor = None
        self.scaler = None
        self.feature_columns = None
        self.is_trained = False
        self.training_metrics = {}
    
    def prepare_data(self, df, target_col='final_score'):
        """Prepare data for training"""
        
        # Initialize preprocessor if not exists
        if self.preprocessor is None:
            self.preprocessor = DataPreprocessor()
            df_processed = self.preprocessor.fit_transform(df, target_col)
        else:
            df_processed = self.preprocessor.transform(df)
        
        # Get feature columns
        self.feature_columns = self.preprocessor.get_feature_columns()
        
        # Prepare X and y
        X = df_processed[self.feature_columns]
        y = df_processed[target_col]
        
        return X, y
    
    def train(self, df, test_size=0.2, random_state=42):
        """Train the model with proper validation"""
        
        print("="*60)
        print("MODEL TRAINING")
        print("="*60)
        
        # Prepare data
        X, y = self.prepare_data(df)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        print(f"\n📊 Data Split:")
        print(f"   Training: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
        print(f"   Testing: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")
        
        # Scale features
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train model
        self.model = LinearRegression()
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_train_pred = self.model.predict(X_train_scaled)
        y_test_pred = self.model.predict(X_test_scaled)
        
        self.training_metrics = {
            'train_r2': r2_score(y_train, y_train_pred),
            'test_r2': r2_score(y_test, y_test_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred)
        }
        
        print(f"\n📈 Performance Metrics:")
        print(f"   Training R²: {self.training_metrics['train_r2']:.4f}")
        print(f"   Testing R²: {self.training_metrics['test_r2']:.4f}")
        print(f"   Testing RMSE: {self.training_metrics['test_rmse']:.2f}")
        print(f"   Testing MAE: {self.training_metrics['test_mae']:.2f}")
        
        # Check for overfitting
        r2_diff = self.training_metrics['train_r2'] - self.training_metrics['test_r2']
        if r2_diff > 0.1:
            print(f"\n⚠️ Possible overfitting detected! (R² diff: {r2_diff:.3f})")
        else:
            print(f"\n✅ No significant overfitting detected")
        
        # Cross-validation
        cv_scores = cross_val_score(self.model, X_train_scaled, y_train, cv=5, scoring='r2')
        print(f"\n📊 Cross-Validation (5-fold):")
        print(f"   Mean R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        self.is_trained = True
        
        # Store test data for later evaluation
        self.X_test = X_test
        self.y_test = y_test
        self.y_test_pred = y_test_pred
        
        return self.model
    
    def predict(self, student_data):
        """
        Make prediction for a single student
        
        Args:
            student_data: dict with feature values
        
        Returns:
            predicted_score: float
        """
        if not self.is_trained:
            raise Exception("Model not trained yet!")
        
        # Convert to DataFrame
        input_df = pd.DataFrame([student_data])
        
        # Preprocess
        input_processed = self.preprocessor.transform(input_df)
        
        # Ensure all features exist
        for col in self.feature_columns:
            if col not in input_processed.columns:
                input_processed[col] = 0
        
        # Select and scale features
        X_input = input_processed[self.feature_columns]
        X_input_scaled = self.scaler.transform(X_input)
        
        # Predict
        prediction = self.model.predict(X_input_scaled)[0]
        
        return prediction
    
    def predict_batch(self, students_data):
        """Make predictions for multiple students"""
        return [self.predict(student) for student in students_data]
    
    def get_feature_importance(self):
        """Get feature importance (coefficients)"""
        if not self.is_trained:
            raise Exception("Model not trained yet!")
        
        importance_df = pd.DataFrame({
            'Feature': self.feature_columns,
            'Coefficient': self.model.coef_
        }).sort_values('Coefficient', key=abs, ascending=False)
        
        return importance_df
    
    def save(self, filepath='student_model.pkl'):
        """Save model and all components"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'model': self.model,
                'preprocessor': self.preprocessor,
                'scaler': self.scaler,
                'feature_columns': self.feature_columns,
                'is_trained': self.is_trained,
                'training_metrics': self.training_metrics
            }, f)
        print(f"✅ Model saved to {filepath}")
    
    def load(self, filepath='student_model.pkl'):
        """Load saved model"""
        with open(filepath, 'rb') as f:
            saved = pickle.load(f)
        
        self.model = saved['model']
        self.preprocessor = saved['preprocessor']
        self.scaler = saved['scaler']
        self.feature_columns = saved['feature_columns']
        self.is_trained = saved['is_trained']
        self.training_metrics = saved['training_metrics']
        print(f"✅ Model loaded from {filepath}")
        return self

# Train the model
model = StudentPerformanceModel()
model.train(df)

MODEL TRAINING
Column Identification:
  Numeric features: ['student_id', 'hours_studied', 'previous_score', 'attendance', 'sleep_hours', 'extracurricular_encoded', 'parent_education_encoded', 'internet_access_encoded']
  Categorical features: ['extracurricular', 'parent_education', 'internet_access']

Encoding categorical variables:
  extracurricular: {'No': np.int64(0), 'Yes': np.int64(1)}
  parent_education: {'Bachelor': np.int64(0), 'High School': np.int64(1), 'Master': np.int64(2), 'PhD': np.int64(3)}
  internet_access: {'No': np.int64(0), 'Yes': np.int64(1)}


DuplicateError: Expected unique column names, got:
- 'extracurricular_encoded' 2 times
- 'parent_education_encoded' 2 times
- 'internet_access_encoded' 2 times